# Task 2.1: Dataset Selection and Setup

## What the dataset is
The dataset is **synthetically generated** for Learning to Rank. We use `sklearn.datasets.make_classification` to produce feature vectors and binary relevance labels, then **group samples by synthetic query IDs**: each "query" has a fixed number of document vectors with relevance labels. Features are continuous; we ensure at least 2 informative features and ≥100 total samples. Train and test splits are made **by query** (e.g. 80% of query IDs for train, 20% for test) so that no query appears in both sets. Features are normalized with `StandardScaler` fitted on the training set only, and the scaler is saved for reproducibility.

## Why it is a reasonable testbed
It mimics **query–document structures** required by the Latent SVM ranking method (Yu & Joachims, 2009, Section 5.3): each "query" has a set of documents with feature vectors and relevance labels; the model learns a weight vector **w** to score documents and optimize Precision@k. The latent variable **h** (e.g. which documents are in the top-k) is not observed; CCCP alternates between inferring **h** and optimizing **w**. The synthetic data provides the same (X, y, qid) structure as LETOR-style benchmarks, so we can run the same evaluation pipeline (NDCG@k, P@k, MAP) and compare baseline vs Latent SVM.

## Limitations compared to the original paper
The original paper uses the **LETOR OHSUMED** benchmark: medical retrieval with 106 queries, 25 hand-crafted ranking features, and graded relevance. Our toy dataset is **smaller** (e.g. ~20 queries, ~200 samples), has **fewer features** (e.g. 5–10), **binary** relevance only, and **no real retrieval semantics**. It is suitable for checking correctness and relative gains of the method, but absolute metrics and conclusions cannot be directly compared to the paper's reported results (e.g. Table 3 in Yu & Joachims, 2009).

## 1. Load Libraries and Set Random Seed

In [5]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries loaded. Random seed set to 42.")

Libraries loaded. Random seed set to 42.


## 2. Generate Synthetic Learning-to-Rank Data

We generate a single pool of samples with `make_classification`, then assign each sample to a synthetic query ID so that we have multiple documents per query (query–document structure).

In [6]:
N_SAMPLES = 200
N_FEATURES = 5
N_INFORMATIVE = 2
DOCS_PER_QUERY = 10
N_QUERIES = N_SAMPLES // DOCS_PER_QUERY
TRAIN_QUERY_RATIO = 0.8
n_train_queries = max(1, int(N_QUERIES * TRAIN_QUERY_RATIO))

X_all, y_all = make_classification(
    n_samples=N_SAMPLES,
    n_features=N_FEATURES,
    n_informative=N_INFORMATIVE,
    n_redundant=0,
    n_clusters_per_class=1,
    random_state=42,
)

qid_all = np.repeat(np.arange(1, N_QUERIES + 1), DOCS_PER_QUERY)
assert len(qid_all) == N_SAMPLES
print(f"Generated {N_SAMPLES} samples, {N_FEATURES} features ({N_INFORMATIVE} informative), {N_QUERIES} queries, {DOCS_PER_QUERY} docs/query.")

Generated 200 samples, 5 features (2 informative), 20 queries, 10 docs/query.


## 3. Split by Query and Build Train/Test Lists

In [7]:
train_queries = np.arange(1, n_train_queries + 1)
test_queries = np.arange(n_train_queries + 1, N_QUERIES + 1)

def build_query_list(query_ids, X_all, y_all, qid_all):
    out = []
    for qid in query_ids:
        mask = qid_all == qid
        out.append({
            'query_id': int(qid),
            'X': X_all[mask].astype(np.float64),
            'y': y_all[mask].astype(np.int64),
        })
    return out

train_data = build_query_list(train_queries, X_all, y_all, qid_all)
test_data = build_query_list(test_queries, X_all, y_all, qid_all)

n_train_samples = sum(d['X'].shape[0] for d in train_data)
n_test_samples = sum(d['X'].shape[0] for d in test_data)
print(f"Train: {len(train_data)} queries, {n_train_samples} samples. Test: {len(test_data)} queries, {n_test_samples} samples.")

Train: 16 queries, 160 samples. Test: 4 queries, 40 samples.


## 4. Normalize Features and Save to partB/data/

In [8]:
X_train_flat = np.vstack([d['X'] for d in train_data])
scaler = StandardScaler()
scaler.fit(X_train_flat)

for d in train_data:
    d['X'] = scaler.transform(d['X'])
for d in test_data:
    d['X'] = scaler.transform(d['X'])

DATA_DIR = Path('data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

np.save(DATA_DIR / 'train_data.npy', train_data, allow_pickle=True)
np.save(DATA_DIR / 'test_data.npy', test_data, allow_pickle=True)
np.save(DATA_DIR / 'scaler.npy', np.array(scaler, dtype=object), allow_pickle=True)

print(f"Saved train_data.npy ({len(train_data)} queries), test_data.npy ({len(test_data)} queries), scaler.npy to partB/data/")

Saved train_data.npy (16 queries), test_data.npy (4 queries), scaler.npy to partB/data/
